# California Housing - Makine Öğrenmesi Projesi

Bu proje California Housing veri seti üzerinde 3 farklı makine öğrenmesi tekniğini uygulayarak tahminde bulunduk bunlar:

1. **Regresyon** - Ev fiyatı tahmini (Random Forest Regressor ve SVR)
2. **Sınıflandırma** - Okyanus yakınlığı sınıflandırması (Random Forest Classifier ve SVM)
3. **Kümeleme** - Gelir ve coğrafi kümeleme (GMM & Agglomerative Clustering ve KMeans)

## Kütüphanelerin Yüklenmesi

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,precision_score, recall_score, f1_score)
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datetime import datetime

print("Kütüphaneler yüklendi!")

## Veri Setinin Yüklenmesi ve Incelenmesi

In [ ]:
df = pd.read_csv('housing.csv')
print(f"Veri seti boyutu: {df.shape[0]} satır, {df.shape[1]} sütun")
df.head(10)

## Veri Seti Temizlenmesi

In [ ]:
def prepare_data(test_size=0.2, random_state=12):
    data2 = data.dropna().copy()
    data2 = data2[~data2.isin([np.inf, -np.inf]).any(axis=1)]
    data2 = data2.reset_index(drop=True)

    return data2

---
# BOLUM 1: REGRESYON
## Ev Fiyati Tahmini - Random Forest Regressor ve Support Vector Regressor

In [ ]:
def prepare_data_regression(test_size=0.2, random_state=12):
    preparedData = prepare_data()

    feature_names = ['longitude', 'latitude', 'median_income','total_rooms','total_bedrooms','population']
    target_name = 'median_house_value'
    
    X = preparedData[feature_names].values
    y = preparedData[target_name].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
    
    return X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test, y_train_scaled, data2, feature_names, scaler_X, scaler_y

In [ ]:
def train_svr_model(X_train_scaled, y_train_scaled, kernel='rbf', C=1.0, epsilon=0.1):
    model = SVR(
        kernel=kernel,
        C=C,
        epsilon=epsilon,
        gamma='scale'
    )
    
    model.fit(X_train_scaled, y_train_scaled)
    
    return model

In [ ]:
def train_rf_model(X_train, y_train, n_estimators=100, max_depth=30):
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=12,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    return model

In [ ]:
X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test, y_train_scaled, data2, feature_names, scaler_X, scaler_y = prepare_data_regression()

svr_model = train_svr_model(X_train_scaled, y_train_scaled)
y_pred_svr_scaled = svr_model.predict(X_test_scaled)
y_pred_svr = scaler_y.inverse_transform(y_pred_svr_scaled.reshape(-1, 1)).ravel()
    
rf_model = train_rf_model(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
    
svr_r2 = r2_score(y_test, y_pred_svr)
svr_mae = mean_absolute_error(y_test, y_pred_svr)
svr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_svr))
    
rf_r2 = r2_score(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    
print(f"\n SVR Performansı:")
print(f"   R² Score: {svr_r2:.4f}")
print(f"   MAE: ${svr_mae:,.0f}")
print(f"   RMSE: ${svr_rmse:,.0f}")
    
print(f"\n Random Forest Performansı:")
print(f"   R² Score: {rf_r2:.4f}")
print(f"   MAE: ${rf_mae:,.0f}")
print(f"   RMSE: ${rf_rmse:,.0f}")
    
print("Grafik oluşturuluyor")
    
images_dir = os.path.join(os.path.dirname(__file__), 'images')
os.makedirs(images_dir, exist_ok=True)
    
print('Grafik oluştu')

metrics = plot_combined_regression_comparison(
    y_test, y_pred_svr, y_pred_rf,
    os.path.join(images_dir, 'svr_rf_comparison.png')
)
    
winner = "SVR" if svr_r2 > rf_r2 else "Random Forest"
diff = abs(svr_r2 - rf_r2)
print(f"\n🏆 En iyi model: {winner} (R² farkı: {diff:.4f})")

---
# BOLUM 2: SINIFLANDIRMA
## Okyanus Yakinligi Siniflandirmasi - Random Forest Classifier ve SVC

In [ ]:
def prepare_data_classification(test_size=0.2, random_state=12):
    preparedData = prepare_data()
    
    feature_names = [
        'longitude', 'latitude', 'housing_median_age', 
        'total_rooms', 'total_bedrooms', 'population', 
        'households', 'median_income', 'median_house_value'
    ]
    target_name = 'ocean_proximity'
    
    X = preparedData[feature_names].values
    y_raw = preparedData[target_name].values

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)
    class_names = label_encoder.classes_

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test, data, label_encoder, feature_names, class_names, scaler

In [ ]:
def train_svm_model(X_train_scaled, y_train, kernel='rbf', C=1.0, gamma='scale'):
    model = SVC(
        kernel=kernel,
        C=C,
        gamma=gamma,
        class_weight='balanced',
        random_state=12
    )
    
    model.fit(X_train_scaled, y_train)
    
    return model

In [ ]:
def train_rf_model(X_train, y_train, n_estimators=100, max_depth=15):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight='balanced',
        random_state=12,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    return model

In [ ]:
X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test, data, label_encoder, feature_names, class_names, scaler = prepare_data_classification()
    
svm_model = train_svm_model(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)
    
rf_model = train_rf_model(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
    
svm_accuracy = accuracy_score(y_test, y_pred_svm)
rf_accuracy = accuracy_score(y_test, y_pred_rf)
    
print(f"\n SVM Test Doğruluğu: {svm_accuracy:.2%}")
print(f" Random Forest Test Doğruluğu: {rf_accuracy:.2%}")
    
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm, target_names=class_names))
    
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=class_names))
    

print('Grafik oluşturuluyor')
images_dir = os.path.join(os.path.dirname(__file__), 'images')
os.makedirs(images_dir, exist_ok=True)
    
print('Grafik oluşturuldu')

metrics = plot_combined_comparison(
    y_test, y_pred_svm, y_pred_rf, class_names,
    os.path.join(images_dir, 'svm_rf_comparison.png')
)
    
winner = "SVM" if svm_accuracy > rf_accuracy else "Random Forest"
diff = abs(svm_accuracy - rf_accuracy)
print(f"\n🏆 En iyi model: {winner} (+{diff:.2%} fark)")


---
# BOLUM 3: KUMELEME
## Gelir Kumeleme (GMM) & Cografi Kumeleme (Haversine) ve KMeans

### 3.1 Gelir Bazli Kumeleme (GMM - Gaussian Mixture Model)

In [ ]:
def train_gmm(data, n_clusters=5):
    income_data = data[['median_income']].values
    
    gmm = GaussianMixture(n_components=n_clusters, random_state=12, n_init=10)
    clusters = gmm.fit_predict(income_data)
    
    cluster_incomes = {i: gmm.means_[i][0] for i in range(n_clusters)}
    sorted_clusters = sorted(cluster_incomes.keys(), key=lambda x: cluster_incomes[x], reverse=True)
    
    return clusters, gmm, sorted_clusters

### 3.1 Gelir Bazli Kumeleme (GMM - Gaussian Mixture Model)

In [ ]:
def train_kmeans(data, n_clusters=5):
    income_data = data[['median_income']].values
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=12, n_init=10)
    clusters = kmeans.fit_predict(income_data)
    
    cluster_incomes = {i: kmeans.cluster_centers_[i][0] for i in range(n_clusters)}
    sorted_clusters = sorted(cluster_incomes.keys(), key=lambda x: cluster_incomes[x], reverse=True)
    
    return clusters, kmeans, sorted_clusters

In [ ]:
# Gelir Haritasi
CLUSTER_COLORS = {'Super Zengin': '#1a5f1a', 'Zengin': '#4CAF50', 'Orta': '#FFC107', 'Fakir': '#FF9800', 'Cok Fakir': '#f44336'}

def get_income_label(cluster_id):
    rank = sorted_clusters.index(cluster_id)
    return INCOME_LABELS[rank] if rank < len(INCOME_LABELS) else f"Kume {cluster_id}"

colors = [CLUSTER_COLORS.get(get_income_label(c), 'gray') for c in clusters_gmm]

plt.figure(figsize=(14, 10))
plt.scatter(cluster_data['longitude'], cluster_data['latitude'], c=colors, alpha=0.6, s=20)
plt.title('California Gelir Haritasi - GMM Kumeleme', fontsize=14, fontweight='bold')
plt.xlabel('Boylam')
plt.ylabel('Enlem')

legend_elements = [Patch(facecolor=CLUSTER_COLORS[label], alpha=0.6, 
                         label=f"{label} (${gmm.means_[sorted_clusters[i]][0]*10000:,.0f})") 
                   for i, label in enumerate(INCOME_LABELS)]
plt.legend(handles=legend_elements, loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()

### 3.2 Cografi Kumeleme (Haversine Distance + Agglomerative)

In [ ]:
def haversine_distance(coord1, coord2):
    R = 6371  # km
    lat1, lon1 = np.radians(coord1[0]), np.radians(coord1[1])
    lat2, lon2 = np.radians(coord2[0]), np.radians(coord2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def haversine_matrix(coords):
    n = len(coords)
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            d = haversine_distance(coords[i], coords[j])
            dist[i, j] = dist[j, i] = d
    return dist

# Ornekleme ile cografi kumeleme
coords = cluster_data[['latitude', 'longitude']].values
sample_size = 2000
sample_idx = np.random.choice(len(coords), sample_size, replace=False)
sample_coords = coords[sample_idx]

dist_matrix = haversine_matrix(sample_coords)
geo_cluster = AgglomerativeClustering(n_clusters=5, metric='precomputed', linkage='average')
sample_clusters = geo_cluster.fit_predict(dist_matrix)

# Merkezleri hesapla
centers = np.array([sample_coords[sample_clusters == i].mean(axis=0) for i in range(5)])

# Tum noktalari en yakin merkeze ata
clusters_geo = np.zeros(len(coords), dtype=int)
for i, coord in enumerate(coords):
    min_dist = float('inf')
    for j, center in enumerate(centers):
        dist = haversine_distance(coord, center)
        if dist < min_dist:
            min_dist = dist
            clusters_geo[i] = j

print("Cografi kumeleme tamamlandi!")

In [ ]:
# Cografi Harita
GEO_COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
colors = [GEO_COLORS[c] for c in clusters_geo]

plt.figure(figsize=(14, 10))
plt.scatter(cluster_data['longitude'], cluster_data['latitude'], c=colors, alpha=0.6, s=20)

for i, center in enumerate(centers):
    plt.scatter(center[1], center[0], c='black', s=200, marker='X', edgecolors='white', linewidths=2, zorder=5)
    plt.annotate(f'Bolge {i}', (center[1], center[0]), textcoords="offset points", 
                xytext=(10, 10), fontweight='bold', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.title('California Cografi Kumeleme - Haversine Distance', fontsize=14, fontweight='bold')
plt.xlabel('Boylam')
plt.ylabel('Enlem')
plt.grid(True, alpha=0.3)
plt.show()

---
# SONUC VE DEGERLENDIRME

In [ ]:
print("="*60)
print("PROJE SONUC OZETI")
print("="*60)
print(f"\nREGRESYON (Ev Fiyati Tahmini):")
print(f"   Model: Random Forest Regressor")
print(f"   R2 Skoru: {r2:.4f}")
print(f"   RMSE: ${rmse:,.0f}")
print(f"\nSINIFLANDIRMA (Okyanus Yakinligi):")
print(f"   Model: Random Forest Classifier")
print(f"   Accuracy: {accuracy:.2%}")
print(f"   F1-Score: {f1:.2%}")
print(f"\nKUMELEME:")
print(f"   GMM: 5 gelir seviyesi belirlendi")
print(f"   Cografi: 5 bolge (Haversine distance)")
print("\n" + "="*60)
print("Proje basariyla tamamlandi!")